In [1]:
import numpy as np
from astropy.table import Table

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


# Load data

In [2]:
diaObj = Table.read("/Users/leo/Projects/PHYS367/alerce/alerce_lsstDiaObj_top100k_ndiasourcesGrt10_19Nov26.fits")
probs  = Table.read("/Users/leo/Projects/PHYS367/alerce/alerce_probs_forTop100k_ndiasourcesGrt10_19Nov26.fits")
taxonomy = Table.read("/Users/leo/Projects/PHYS367/alerce/alerce_taxonomy.fits")

# Match objects

In [3]:
probsOid = probs["oid"]
diaObjOid = diaObj["oid"]
oid_matched = np.intersect1d(probsOid.value.data, diaObjOid.value.data)
nMatched = len(oid_matched)
print(f"There are {nMatched:,} sources in the object catalog that have an entry in the probability catalog.")

There are 69,231 sources in the object catalog that have an entry in the probability catalog.


In [ ]:
def getProbabilities(i):
    evtClass = probs[probs["oid"] == diaObj["oid"][i]]
    evtProbs = dict()
    
    for classifier in np.unique(evtClass["classifier_id"]):
        evtProbs[classifier] = dict()
        for id in evtClass["class_id"][evtClass["classifier_id"] == classifier]:
            name = taxonomy[(taxonomy["class_id"] == id) & (taxonomy["classifier_id"] == classifier)]["class_name"]
            if len(name) == 1:
                evtProbs[classifier][name[0]] = evtClass[(evtClass["classifier_id"] == classifier) & (evtClass["class_id"] == id)]["probability"][0]
    
    return evtProbs

In [ ]:
combined = Table(names = [
    "oid", "ra", "dec", "probabilities"
], dtype = [int, np.float64, np.float64, object])

In [ ]:
for i, row in enumerate(diaObj):
    combined.add_row({"oid": row["oid"], "ra": row["ra"], "dec": row["dec"], "probabilities": getProbabilities(i)})

In [ ]:
combined